# Загрузка официальных курсов ЦБ РФ

Ноутбук загружает дневную историю с 01.01.2020, сохраняет исходный temporal-слой и формирует широкий DataFrame: строки — даты доступности, столбцы — валюты, значения — рубли за одну единицу валюты.


In [ ]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cbr_loader import CURRENCIES, load_cbr_history

START_DATE = date(2020, 1, 1)
END_DATE = date.today()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cbr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
cbr_history = load_cbr_history(
    start_date=START_DATE,
    end_date=END_DATE,
    currencies=CURRENCIES,
    raw_dir=RAW_DIR,
)

assert cbr_history["normalized_rate"].gt(0).all()
assert cbr_history["publication_timestamp_is_proxy"].all()
assert (
    cbr_history["available_at"]
    == cbr_history["effective_date"] + pd.Timedelta(days=1)
).all()
pd.testing.assert_series_equal(
    cbr_history["normalized_rate"],
    cbr_history["raw_rate"] / cbr_history["nominal"],
    check_names=False,
)

rates = (
    cbr_history
    .pivot(index="available_at", columns="currency", values="normalized_rate")
    .reindex(columns=list(CURRENCIES))
    .sort_index()
)
rates.columns.name = None
rates.index.name = "available_at"
assert rates.notna().all().all()
assert rates.index.is_unique and rates.index.is_monotonic_increasing

cbr_history.to_parquet(
    PROCESSED_DIR / "cbr_history_temporal.parquet",
    index=False,
)
rates.to_parquet(PROCESSED_DIR / "cbr_rates.parquet")
rates.to_csv(PROCESSED_DIR / "cbr_rates.csv")

pd.DataFrame({
    "publication_dates": [len(rates)],
    "first_available_at": [rates.index.min()],
    "last_available_at": [rates.index.max()],
    "currencies": [", ".join(rates.columns)],
})


In [ ]:
rates.tail()
